In [ ]:
import json

import great_expectations as gx
import pandas as pd
import synapseclient
from synapseclient.models import File

from agoradatatools.gx import GreatExpectationsRunner
from notebook_setup import configure_notebook_output

# Suppresses logging spam; must run before gx.get_context() to suppress parseString deprecation warning from GX
configure_notebook_output()

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members import ExpectColumnValuesToHaveListMembers
from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_unique_list_members import ExpectColumnValuesToHaveUniqueListMembers
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange
from expectations.expect_column_nested_field_values_to_be_mostly_not_null import ExpectColumnNestedObjectNotNull
from expectations.expect_column_nested_field_values_mostly_meet_string_requirement import ExpectColumnNestedObjectStringLength
from expectations.expect_column_nested_field_values_mostly_regex import ExpectColumnNestedObjectRegexRule

# Create Expectation Suite for Gene Info Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# preprod
gene_info_entity = await File(id="syn17015359").get_async()
gene_info_data_file = gene_info_entity.path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(gene_info_data_file)
nested_columns = ['target_nominations', 'median_expression', 'druggability', 'ensembl_info']
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "gene_info"

## Add Expectations to Validator Object For Each Column

In [ ]:
# ensembl_gene_id
validator.expect_column_values_to_be_of_type("ensembl_gene_id", "str")
validator.expect_column_values_to_not_be_null("ensembl_gene_id")
validator.expect_column_values_to_match_regex("ensembl_gene_id", "^ENSG[0-9]{11}$")
validator.expect_column_values_to_be_unique("ensembl_gene_id")

In [ ]:
# name
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_value_lengths_to_be_between("name", 1, 200)
# completeness floor: ~79.4% non-null in current data
validator.expect_column_values_to_not_be_null("name", mostly=0.75)

In [ ]:
# summary
validator.expect_column_values_to_be_of_type("summary", "str")
validator.expect_column_values_to_not_be_null("summary", mostly=0.50)

In [ ]:
# hgnc_symbol
validator.expect_column_values_to_be_of_type("hgnc_symbol", "str")
validator.expect_column_value_lengths_to_be_between("hgnc_symbol", 1, 25)
validator.expect_column_values_to_match_regex("hgnc_symbol", "^[a-zA-Z0-9_.-]*$")
# completeness floor: ~79.4% non-null in current data
validator.expect_column_values_to_not_be_null("hgnc_symbol", mostly=0.75)

In [ ]:
# alias
validator.expect_column_values_to_be_of_type("alias", "list")
validator.expect_column_values_to_not_be_null("alias")
validator.expect_column_values_to_have_list_members_of_type(column="alias", member_type="str")
validator.expect_column_values_to_have_unique_list_members(column="alias")
validator.expect_column_values_to_have_list_length_in_range(column="alias", list_length_range=[0, 23])

In [ ]:
# is_igap
validator.expect_column_values_to_be_of_type("is_igap", "bool")
validator.expect_column_values_to_not_be_null("is_igap")

In [ ]:
# is_eqtl
validator.expect_column_values_to_be_of_type("is_eqtl", "bool")
validator.expect_column_values_to_not_be_null("is_eqtl")

In [ ]:
# is_any_rna_changed_in_ad_brain
validator.expect_column_values_to_be_of_type("is_any_rna_changed_in_ad_brain", "bool")
validator.expect_column_values_to_not_be_null("is_any_rna_changed_in_ad_brain")

In [ ]:
# rna_brain_change_studied
validator.expect_column_values_to_be_of_type("rna_brain_change_studied", "bool")
validator.expect_column_values_to_not_be_null("rna_brain_change_studied")

In [ ]:
# is_any_protein_changed_in_ad_brain
validator.expect_column_values_to_be_of_type("is_any_protein_changed_in_ad_brain", "bool")
validator.expect_column_values_to_not_be_null("is_any_protein_changed_in_ad_brain")

In [ ]:
# protein_brain_change_studied
validator.expect_column_values_to_be_of_type("protein_brain_change_studied", "bool")
validator.expect_column_values_to_not_be_null("protein_brain_change_studied")

In [ ]:
# target_nominations
with open("../src/agoradatatools/great_expectations/gx/json_schemas/gene_info/target_nominations.json", "r") as file:
    target_nominations_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("target_nominations", json_schema=target_nominations_schema)

In [ ]:
# target_nominations
validator.expect_column_nested_object_not_null(column="target_nominations", non_null_threshold=0.85, target_field="validation_study_details")

In [ ]:
# target_nominations
# data_used_to_support_target_selection is nullable per-item in the schema; enforce a collection-level
# density floor: >=95% of nomination objects must carry a non-empty string (something JSON Schema can't express).
validator.expect_column_nested_object_string_length(
    column="target_nominations",
    target_field="data_used_to_support_target_selection",
    operator=">",
    length_threshold=0,
    valid_threshold=0.95,
)

In [ ]:
# target_nominations
# Nested-object density floor a per-document JSON Schema can't express: >=70% of nominations carry a
# valid Synapse ID (observed ~73.7% non-null and matching ^syn[0-9]+$).
# (A second nested_object_not_null on this column is not possible — GX keys aggregate expectations by
# type+column, so it would overwrite the existing validation_study_details floor.)
validator.expect_column_nested_object_regex_rule(
    column="target_nominations",
    target_field="data_synapseid",
    regex_pattern="^syn[0-9]+$",
    valid_threshold=0.70,
)

In [ ]:
# median_expression
with open("../src/agoradatatools/great_expectations/gx/json_schemas/gene_info/median_expression.json", "r") as file:
    median_expression_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("median_expression", json_schema=median_expression_schema)
# Coverage floor: ~68.0% of genes have median expression ((gene_metadata:syn25953363.17 - median_expression:syn27211878.2)
validator.expect_column_values_to_not_be_in_set("median_expression", ["null", "NaN"], mostly=0.64)

In [ ]:
# druggability
with open("../src/agoradatatools/great_expectations/gx/json_schemas/gene_info/druggability.json", "r") as file:
    druggability_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("druggability", json_schema=druggability_schema)

In [ ]:
# total_nominations
validator.expect_column_values_to_be_of_type("total_nominations", "float")
validator.expect_column_values_to_be_between("total_nominations", min_value=1, max_value=10)

In [ ]:
# biodomains
validator.expect_column_values_to_be_of_type("biodomains", "list")
# Coverage floor: ~44.6% of genes have biodomains (gene_metadata:syn25953363.17] - genes_biodmains:syn44151254.5)
validator.expect_column_values_to_not_be_null("biodomains", mostly=.40)
validator.expect_column_values_to_have_list_length_in_range(column="biodomains", list_length_range=[1, 19])
validator.expect_column_values_to_have_unique_list_members(column="biodomains")
validator.expect_column_values_to_have_list_members(column="biodomains", list_members=sorted([
        'Apoptosis',
        'Vasculature',
        'Lipid Metabolism',
        'Metal Binding and Homeostasis',
        'Proteostasis',
        'Immune Response',
        'Cell Cycle',
        'DNA Repair',
        'Autophagy',
        'Mitochondrial Metabolism',
        'Structural Stabilization',
        'Synapse',
        'Endolysosome',
        'Oxidative Stress',
        'Epigenetic',
        'APP Metabolism',
        'RNA Spliceosome',
        'Tau Homeostasis',
        'Myelination'
    ])
)

In [ ]:
# is_adi
validator.expect_column_values_to_be_of_type("is_adi", "bool")
validator.expect_column_values_to_not_be_null("is_adi")

In [ ]:
# is_tep
validator.expect_column_values_to_be_of_type("is_tep", "bool")
validator.expect_column_values_to_not_be_null("is_tep")

In [ ]:
# resource_url
validator.expect_column_values_to_be_of_type("resource_url", "str")
validator.expect_column_values_to_match_regex(
    "resource_url", r"^https://adknowledgeportal\.synapse\.org/Explore/Target%20Enabling%20Resources\?"
)

In [ ]:
# ensembl_info
with open("../src/agoradatatools/great_expectations/gx/json_schemas/gene_info/ensembl_info.json", "r") as file:
    ensembl_info_schema = json.load(file)

validator.expect_column_values_to_match_json_schema("ensembl_info", json_schema=ensembl_info_schema)

In [ ]:
# uniprotkb_accessions
validator.expect_column_values_to_be_of_type("uniprotkb_accessions", "list")
# Coverage floor: ~51.7% of genes have UniProtKB accessions (gene_metadata:syn25953363.17] - ensg_to_uniprot_mapping:syn54113663.7)
validator.expect_column_values_to_not_be_null("uniprotkb_accessions", mostly=0.47)
validator.expect_column_values_to_have_list_members_of_type(column="uniprotkb_accessions", member_type="str")
validator.expect_column_values_to_have_unique_list_members(column="uniprotkb_accessions")
validator.expect_column_values_to_have_list_length_in_range(column="uniprotkb_accessions", list_length_range=[1, 4])

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()